In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_train_clean3.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_val_clean3.parquet')

print(train.shape, val.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(616293, 35) (133316, 35)


In [2]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [6]:
# ============================================================
# 1. 데이터 불러오기 및 Tidal / Normal 분리
# ============================================================

# 1-2. 조석 주기성 후보 관측소 9개
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# 1-3. Train 분리
tidal_train = train.loc[
    train['station_id'].isin(tidal_stations)
].copy()

normal_train = train.loc[
    ~train['station_id'].isin(tidal_stations)
].copy()

# 1-4. Validation 분리
tidal_val = val.loc[
    val['station_id'].isin(tidal_stations)
].copy()

normal_val = val.loc[
    ~val['station_id'].isin(tidal_stations)
].copy()

# 1-5. 데이터 분리 검증
assert len(tidal_train) + len(normal_train) == len(train)
assert len(tidal_val) + len(normal_val) == len(val)

print(f'Train: {len(train):,}')
print(f'  Tidal: {len(tidal_train):,}')
print(f'  Normal: {len(normal_train):,}')

print(f'\nValidation: {len(val):,}')
print(f'  Tidal: {len(tidal_val):,}')
print(f'  Normal: {len(normal_val):,}')

Train: 616,293
  Tidal: 11,685
  Normal: 604,608

Validation: 133,316
  Tidal: 2,566
  Normal: 130,750


In [7]:
# ============================================================
# 2. Tidal / Normal 독립변수 및 종속변수 분리
# ============================================================

# 2-1. Tidal
X_tidal_train = tidal_train[num_cols_fe + ['station_id']].copy()
y_tidal_train = tidal_train['target_maxrise_6h']

# 2-2. Normal
X_normal_train = normal_train[num_cols_fe + ['station_id']].copy()
y_normal_train = normal_train['target_maxrise_6h']

# 2-3. CatBoost 범주형 변수 처리
X_tidal_train['station_id'] = X_tidal_train['station_id'].astype(str)
X_normal_train['station_id'] = X_normal_train['station_id'].astype(str)

In [8]:
# ============================================================
# 3. 시간사상 Group K-Fold 설정: 분할 전 적적성 검증
# ============================================================


# ------------------------------------------------------------
# 3-1. Group 설정
# 같은 시간사상에 속한 관측치는 반드시 같은 fold에 들어가도록 설정
# ------------------------------------------------------------

# 실제 시간사상 ID 컬럼명
kf_groups = train['event_id']

print(f'<시간사상 Group 확인>')
print(f'전체 학습 행 수   : {len(train):,}')
print(f'시간사상 개수     : {kf_groups.nunique():,}')
print()


# ------------------------------------------------------------
# 3-2. 기존 OOF에서 추출한 사상별 fold 배정 불러오기
# ------------------------------------------------------------

N_SPLITS = 5  # 기존 5-Fold 유지

kf_fold_map = pd.read_csv(
    PROJECT_ROOT / 'config' / '0_kf_event_fold_match.csv'
)
# 새로운 사상/누락/중복이 있으면 재분할하지 않고 중단한다.
assert kf_fold_map.columns.tolist() == ['event_id', 'fold_num']
assert not kf_fold_map.isna().any().any()
assert kf_fold_map['event_id'].is_unique
assert set(kf_fold_map['fold_num']) == set(range(1, N_SPLITS + 1))
assert not kf_groups.isna().any()
assert set(kf_groups) == set(kf_fold_map['event_id'])

kf_fold_num = kf_groups.map(kf_fold_map.set_index('event_id')['fold_num'])
assert kf_fold_num.notna().all()

# 행 위치 배열만 보관한다. 데이터셋은 복사하지 않는다.
# 학습·진단·OOF fold 기록에서 모두 같은 분할을 재사용한다.
kf_splits = [
    (
        np.flatnonzero(kf_fold_num.to_numpy() != fold),
        np.flatnonzero(kf_fold_num.to_numpy() == fold)
    )
    for fold in range(1, N_SPLITS + 1)
]

# ------------------------------------------------------------
# 3-3. Fold 검증
# ------------------------------------------------------------

for fold, (train_idx, valid_idx) in enumerate(
    kf_splits,
    start=1
):
    # 고정한 분할의 사상 중복 여부 검증
    train_groups = set(kf_groups.iloc[train_idx])  # 분할된 event_id
    valid_groups = set(kf_groups.iloc[valid_idx])  # 분할된 event_id

    overlap = train_groups & valid_groups  # 교집합은 0이어야함

    print(
        f'Fold {fold} | '
        f'train={len(train_idx):,} | '
        f'valid={len(valid_idx):,} | '
        f'train events={len(train_groups):,} | '
        f'valid events={len(valid_groups):,} | '
        f'event overlap={len(overlap)}'
    )

<시간사상 Group 확인>
전체 학습 행 수   : 616,293
시간사상 개수     : 98

Fold 1 | train=493,297 | valid=122,996 | train events=82 | valid events=16 | event overlap=0
Fold 2 | train=492,894 | valid=123,399 | train events=77 | valid events=21 | event overlap=0
Fold 3 | train=492,894 | valid=123,399 | train events=77 | valid events=21 | event overlap=0
Fold 4 | train=493,077 | valid=123,216 | train events=78 | valid events=20 | event overlap=0
Fold 5 | train=493,010 | valid=123,283 | train events=78 | valid events=20 | event overlap=0


In [9]:
# ============================================================
# 3-4. Tidal / Normal 고정 Fold 분할 및 검증
# ============================================================

# 1. 기존 고정 Fold 번호 적용
tidal_fold_num = tidal_train['event_id'].map(
    kf_fold_map.set_index('event_id')['fold_num']
)

normal_fold_num = normal_train['event_id'].map(
    kf_fold_map.set_index('event_id')['fold_num']
)

# Fold 매핑 누락 확인
assert tidal_fold_num.notna().all()
assert normal_fold_num.notna().all()

# 2. 각각의 Fold 인덱스 생성
tidal_kf_splits = [
    (
        np.flatnonzero(tidal_fold_num.to_numpy() != fold),
        np.flatnonzero(tidal_fold_num.to_numpy() == fold)
    )
    for fold in range(1, N_SPLITS + 1)
]

normal_kf_splits = [
    (
        np.flatnonzero(normal_fold_num.to_numpy() != fold),
        np.flatnonzero(normal_fold_num.to_numpy() == fold)
    )
    for fold in range(1, N_SPLITS + 1)
]

# 3. Fold별 데이터 개수 및 사상 중복 검증
for name, df, splits in [
    ('Tidal', tidal_train, tidal_kf_splits),
    ('Normal', normal_train, normal_kf_splits)
]:
    print(f'\n<{name} GroupKFold 검증>')

    for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
        train_groups = set(df['event_id'].iloc[train_idx])
        valid_groups = set(df['event_id'].iloc[valid_idx])

        overlap = train_groups & valid_groups

        assert len(train_idx) > 0
        assert len(valid_idx) > 0
        assert len(overlap) == 0

        print(
            f'Fold {fold} | '
            f'train={len(train_idx):,} | '
            f'valid={len(valid_idx):,} | '
            f'train events={len(train_groups):,} | '
            f'valid events={len(valid_groups):,} | '
            f'event overlap={len(overlap)}'
        )


<Tidal GroupKFold 검증>
Fold 1 | train=9,308 | valid=2,377 | train events=82 | valid events=16 | event overlap=0
Fold 2 | train=9,388 | valid=2,297 | train events=77 | valid events=21 | event overlap=0
Fold 3 | train=9,338 | valid=2,347 | train events=77 | valid events=21 | event overlap=0
Fold 4 | train=9,347 | valid=2,338 | train events=78 | valid events=20 | event overlap=0
Fold 5 | train=9,359 | valid=2,326 | train events=78 | valid events=20 | event overlap=0

<Normal GroupKFold 검증>
Fold 1 | train=483,989 | valid=120,619 | train events=82 | valid events=16 | event overlap=0
Fold 2 | train=483,506 | valid=121,102 | train events=77 | valid events=21 | event overlap=0
Fold 3 | train=483,556 | valid=121,052 | train events=77 | valid events=21 | event overlap=0
Fold 4 | train=483,730 | valid=120,878 | train events=78 | valid events=20 | event overlap=0
Fold 5 | train=483,651 | valid=120,957 | train events=78 | valid events=20 | event overlap=0


In [10]:
# ============================================================
# 4. Tidal / Normal GroupKFold 학습 및 OOF 생성
# ============================================================

# 모델별 결과 저장
kf_results = []
tidal_kf_oof_pred = np.full(len(X_tidal_train), np.nan)
normal_kf_oof_pred = np.full(len(X_normal_train), np.nan)

# Tidal / Normal 각각 5-Fold 학습
for name, X, y, splits, oof_pred in [
    ('Tidal', X_tidal_train, y_tidal_train,
     tidal_kf_splits, tidal_kf_oof_pred),

    ('Normal', X_normal_train, y_normal_train,
     normal_kf_splits, normal_kf_oof_pred)
]:

    print(f'\n========== {name} 5-Fold 학습 ==========')

    for fold_num, (train_idx, val_idx) in enumerate(
        splits,
        start=1
    ):
        # Fold별 데이터 분할
        X_kf_tr = X.iloc[train_idx]
        X_kf_va = X.iloc[val_idx]

        y_kf_tr = y.iloc[train_idx]
        y_kf_va = y.iloc[val_idx]

        # 모델 정의
        model_cat_kf = CatBoostRegressor(
            random_state=42,
            iterations=3000,
            cat_features=['station_id']
        )

        # 학습
        model_cat_kf.fit(
            X_kf_tr,
            y_kf_tr,
            early_stopping_rounds=100,
            eval_set=(X_kf_va, y_kf_va),
            verbose=200
        )

        # 예측
        pred_kf_va = model_cat_kf.predict(X_kf_va)
        pred_kf_tr = model_cat_kf.predict(X_kf_tr)

        oof_pred[val_idx] = pred_kf_va

        # 평가
        val_rmse = root_mean_squared_error(y_kf_va, pred_kf_va)
        train_rmse = root_mean_squared_error(y_kf_tr, pred_kf_tr)

        # 결과 저장
        kf_results.append({
            'model': name,
            'fold': fold_num,
            'val_rmse': val_rmse,
            'train_rmse': train_rmse,
            'gap': val_rmse - train_rmse,
            'best_iteration': model_cat_kf.best_iteration_
        })

        print(
            f'{name} Fold {fold_num} | '
            f'Val={val_rmse:.4f} | '
            f'Train={train_rmse:.4f} | '
            f'Gap={val_rmse - train_rmse:.4f} | '
            f'Best iter={model_cat_kf.best_iteration_}'
        )

    # 모델별 전체 OOF RMSE
    assert not np.isnan(oof_pred).any()

    print(
        f'\n{name} OOF RMSE: '
        f'{root_mean_squared_error(y, oof_pred):.6f}'
    )

# Fold별 결과표
kf_results = pd.DataFrame(kf_results)

display(kf_results)


========== Tidal 5-Fold 학습 ==========
Learning rate set to 0.03698
0:	learn: 1.6741950	test: 1.8746893	best: 1.8746893 (0)	total: 183ms	remaining: 9m 9s
200:	learn: 0.3966562	test: 0.5275649	best: 0.5275649 (200)	total: 9.65s	remaining: 2m 14s
400:	learn: 0.3374297	test: 0.4975205	best: 0.4975205 (400)	total: 19.1s	remaining: 2m 4s
600:	learn: 0.3016497	test: 0.4840770	best: 0.4840770 (600)	total: 29.1s	remaining: 1m 56s
800:	learn: 0.2775717	test: 0.4791114	best: 0.4788893 (770)	total: 38.6s	remaining: 1m 45s
1000:	learn: 0.2575283	test: 0.4771059	best: 0.4768975 (947)	total: 46.9s	remaining: 1m 33s
1200:	learn: 0.2401385	test: 0.4761404	best: 0.4751086 (1169)	total: 55.4s	remaining: 1m 22s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.4751086213
bestIteration = 1169

Shrink model to first 1170 iterations.
Tidal Fold 1 | Val=0.4751 | Train=0.2427 | Gap=0.2324 | Best iter=1169
Learning rate set to 0.037029
0:	learn: 1.7144391	test: 1.7292361	best: 1.7292361 (0)	

,model,fold,val_rmse,train_rmse,gap,best_iteration
0,Tidal,1,0.475109,0.242733,0.232376,1169
1,Tidal,2,0.359429,0.211879,0.147550,1651
2,Tidal,3,0.344497,0.154577,0.189920,2975
3,Tidal,4,0.372886,0.157071,0.215815,2918
4,Tidal,5,0.363554,0.159285,0.204270,2819
5,Normal,1,0.264968,0.185222,0.079746,690
6,Normal,2,0.138159,0.206136,-0.067977,896
7,Normal,3,0.222469,0.174849,0.047620,2153
8,Normal,4,0.268136,0.187453,0.080682,566
9,Normal,5,0.214383,0.189199,0.025184,1118


In [11]:
# ============================================================
# 5. Tidal + Normal OOF 예측값 결합 및 전체 RMSE
# ============================================================

# 1. OOF 예측값 결합
kf_oof_pred = pd.Series(index=train.index, dtype=float)

kf_oof_pred.loc[tidal_train.index] = tidal_kf_oof_pred
kf_oof_pred.loc[normal_train.index] = normal_kf_oof_pred

# 2. 예측 누락 검증
assert kf_oof_pred.notna().all()

# 3. 전체 OOF RMSE
oof_rmse = root_mean_squared_error(
    train['target_maxrise_6h'],
    kf_oof_pred
)

print(f'Tidal OOF RMSE  : {root_mean_squared_error(y_tidal_train, tidal_kf_oof_pred):.6f}')
print(f'Normal OOF RMSE : {root_mean_squared_error(y_normal_train, normal_kf_oof_pred):.6f}')
print(f'전체 OOF RMSE   : {oof_rmse:.6f}')

Tidal OOF RMSE  : 0.386370
Normal OOF RMSE : 0.226512
전체 OOF RMSE   : 0.230576


In [ ]:
# ============================================================
# 6. Tidal 모델 하이퍼파라미터 실험 (depth=4)
# ============================================================

# 1. OOF 예측값 및 결과 저장
tidal_kf_oof_pred_d4 = np.full(len(X_tidal_train), np.nan)
tidal_kf_results_d4 = []

# 2. Tidal 5-Fold 학습
for fold_num, (train_idx, val_idx) in enumerate(
    tidal_kf_splits,
    start=1
):
    # Fold별 데이터 분할
    X_kf_tr = X_tidal_train.iloc[train_idx]
    X_kf_va = X_tidal_train.iloc[val_idx]

    y_kf_tr = y_tidal_train.iloc[train_idx]
    y_kf_va = y_tidal_train.iloc[val_idx]

    # 모델 정의 (depth=4)
    model_tidal_d4 = CatBoostRegressor(
        random_state=42,
        iterations=3000,
        depth=4,
        cat_features=['station_id']
    )

    # 학습
    model_tidal_d4.fit(
        X_kf_tr,
        y_kf_tr,
        early_stopping_rounds=100,
        eval_set=(X_kf_va, y_kf_va),
        verbose=200
    )

    # 예측
    pred_kf_va = model_tidal_d4.predict(X_kf_va)
    pred_kf_tr = model_tidal_d4.predict(X_kf_tr)

    tidal_kf_oof_pred_d4[val_idx] = pred_kf_va

    # RMSE 계산
    val_rmse = root_mean_squared_error(y_kf_va, pred_kf_va)
    train_rmse = root_mean_squared_error(y_kf_tr, pred_kf_tr)

    tidal_kf_results_d4.append({
        'fold': fold_num,
        'val_rmse': val_rmse,
        'train_rmse': train_rmse,
        'gap': val_rmse - train_rmse,
        'best_iteration': model_tidal_d4.best_iteration_
    })

    print(
        f'Fold {fold_num} | '
        f'Val={val_rmse:.4f} | '
        f'Train={train_rmse:.4f} | '
        f'Gap={val_rmse - train_rmse:.4f} | '
        f'Best iter={model_tidal_d4.best_iteration_}'
    )

# 3. Fold별 결과표
tidal_kf_results_d4 = pd.DataFrame(tidal_kf_results_d4)

display(tidal_kf_results_d4)

# 4. Tidal OOF RMSE
assert not np.isnan(tidal_kf_oof_pred_d4).any()

rmse_tidal_d4 = root_mean_squared_error(
    y_tidal_train,
    tidal_kf_oof_pred_d4
)

# 5. 기존 Normal OOF와 결합
kf_oof_pred_d4 = pd.Series(index=train.index, dtype=float)

kf_oof_pred_d4.loc[tidal_train.index] = tidal_kf_oof_pred_d4
kf_oof_pred_d4.loc[normal_train.index] = normal_kf_oof_pred

assert kf_oof_pred_d4.notna().all()

# 6. 전체 OOF RMSE
rmse_split_d4 = root_mean_squared_error(
    train['target_maxrise_6h'],
    kf_oof_pred_d4
)

# 7. 기존 모델과 비교
print('\n========== OOF RMSE 비교 ==========')

print(f'Tidal 기존 depth=6 : {root_mean_squared_error(y_tidal_train, tidal_kf_oof_pred):.6f}')
print(f'Tidal 신규 depth=4 : {rmse_tidal_d4:.6f}')
print()
print(f'전체 기존 depth=6 : {oof_rmse:.6f}')
print(f'전체 신규 depth=4 : {rmse_split_d4:.6f}')
print(f'전체 RMSE 변화    : {rmse_split_d4 - oof_rmse:+.6f}')

Learning rate set to 0.03698
0:	learn: 1.6779522	test: 1.8784777	best: 1.8784777 (0)	total: 22.6ms	remaining: 1m 7s
200:	learn: 0.4406629	test: 0.5640272	best: 0.5640272 (200)	total: 5.39s	remaining: 1m 15s
400:	learn: 0.3870816	test: 0.5270947	best: 0.5270947 (400)	total: 10.7s	remaining: 1m 9s
600:	learn: 0.3564261	test: 0.5131287	best: 0.5126495 (592)	total: 16s	remaining: 1m 3s
800:	learn: 0.3348183	test: 0.5048031	best: 0.5048031 (800)	total: 22.2s	remaining: 1m
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.5020844264
bestIteration = 870

Shrink model to first 871 iterations.
Fold 1 | Val=0.5021 | Train=0.3293 | Gap=0.1728 | Best iter=870
Learning rate set to 0.037029
0:	learn: 1.7164441	test: 1.7319029	best: 1.7319029 (0)	total: 25.9ms	remaining: 1m 17s
200:	learn: 0.4590079	test: 0.4567139	best: 0.4567139 (200)	total: 6.34s	remaining: 1m 28s
400:	learn: 0.4022640	test: 0.4246276	best: 0.4246276 (400)	total: 11.4s	remaining: 1m 13s
600:	learn: 0.3678627	tes

,fold,val_rmse,train_rmse,gap,best_iteration
0,1,0.502084,0.329272,0.172812,870
1,2,0.371577,0.255412,0.116165,2469
2,3,0.365325,0.243951,0.121374,2684
3,4,0.381528,0.236899,0.144629,2993
4,5,0.373610,0.236252,0.137357,2999



========== OOF RMSE 비교 ==========
Tidal 기존 depth=6 : 0.386370
Tidal 신규 depth=4 : 0.402665

전체 기존 depth=6 : 0.230576
전체 신규 depth=4 : 0.231104
전체 RMSE 변화    : +0.000528


In [13]:
# ============================================================
# 6. Tidal 모델 하이퍼파라미터 실험 (l2_leaf_reg=10)
# ============================================================

# 1. OOF 예측값 및 결과 저장
tidal_kf_oof_pred_l2 = np.full(len(X_tidal_train), np.nan)
tidal_kf_results_l2 = []

# 2. Tidal 5-Fold 학습
for fold_num, (train_idx, val_idx) in enumerate(
    tidal_kf_splits,
    start=1
):
    # Fold별 데이터 분할
    X_kf_tr = X_tidal_train.iloc[train_idx]
    X_kf_va = X_tidal_train.iloc[val_idx]

    y_kf_tr = y_tidal_train.iloc[train_idx]
    y_kf_va = y_tidal_train.iloc[val_idx]

    # 모델 정의 (l2_leaf_reg=10)
    model_tidal_l2 = CatBoostRegressor(
        random_state=42,
        iterations=3000,
        depth=6,
        l2_leaf_reg=10,
        cat_features=['station_id']
    )

    # 학습
    model_tidal_l2.fit(
        X_kf_tr,
        y_kf_tr,
        early_stopping_rounds=100,
        eval_set=(X_kf_va, y_kf_va),
        verbose=200
    )

    # 예측
    pred_kf_va = model_tidal_l2.predict(X_kf_va)
    pred_kf_tr = model_tidal_l2.predict(X_kf_tr)

    tidal_kf_oof_pred_l2[val_idx] = pred_kf_va

    # RMSE 계산
    val_rmse = root_mean_squared_error(y_kf_va, pred_kf_va)
    train_rmse = root_mean_squared_error(y_kf_tr, pred_kf_tr)

    tidal_kf_results_l2.append({
        'fold': fold_num,
        'val_rmse': val_rmse,
        'train_rmse': train_rmse,
        'gap': val_rmse - train_rmse,
        'best_iteration': model_tidal_l2.best_iteration_
    })

    print(
        f'Fold {fold_num} | '
        f'Val={val_rmse:.4f} | '
        f'Train={train_rmse:.4f} | '
        f'Gap={val_rmse - train_rmse:.4f} | '
        f'Best iter={model_tidal_l2.best_iteration_}'
    )

# 3. Fold별 결과표
tidal_kf_results_l2 = pd.DataFrame(tidal_kf_results_l2)

display(tidal_kf_results_l2)

# 4. Tidal OOF RMSE
assert not np.isnan(tidal_kf_oof_pred_l2).any()

rmse_tidal_l2 = root_mean_squared_error(
    y_tidal_train,
    tidal_kf_oof_pred_l2
)

# 5. 기존 Normal OOF와 결합
kf_oof_pred_l2 = pd.Series(index=train.index, dtype=float)

kf_oof_pred_l2.loc[tidal_train.index] = tidal_kf_oof_pred_l2
kf_oof_pred_l2.loc[normal_train.index] = normal_kf_oof_pred

assert kf_oof_pred_l2.notna().all()

# 6. 전체 OOF RMSE
rmse_split_l2 = root_mean_squared_error(
    train['target_maxrise_6h'],
    kf_oof_pred_l2
)

# 7. 기존 모델과 비교
print('\n========== OOF RMSE 비교 ==========')

print(f'Tidal 기존 l2=3  : {root_mean_squared_error(y_tidal_train, tidal_kf_oof_pred):.6f}')
print(f'Tidal 신규 l2=10 : {rmse_tidal_l2:.6f}')
print()
print(f'전체 기존 l2=3  : {oof_rmse:.6f}')
print(f'전체 신규 l2=10 : {rmse_split_l2:.6f}')
print(f'전체 RMSE 변화  : {rmse_split_l2 - oof_rmse:+.6f}')

0:	learn: 1.6854482	test: 1.8865116	best: 1.8865116 (0)	total: 38.5ms	remaining: 1m 55s
200:	learn: 0.4186804	test: 0.5496595	best: 0.5496595 (200)	total: 8.44s	remaining: 1m 57s
400:	learn: 0.3673639	test: 0.5148383	best: 0.5147764 (399)	total: 17s	remaining: 1m 49s
600:	learn: 0.3332448	test: 0.4987380	best: 0.4987380 (600)	total: 25.7s	remaining: 1m 42s
800:	learn: 0.3108613	test: 0.4904713	best: 0.4904713 (800)	total: 34.7s	remaining: 1m 35s
1000:	learn: 0.2923051	test: 0.4849095	best: 0.4849095 (1000)	total: 43.5s	remaining: 1m 26s
1200:	learn: 0.2789588	test: 0.4825866	best: 0.4825866 (1200)	total: 52s	remaining: 1m 17s
1400:	learn: 0.2682753	test: 0.4792373	best: 0.4792373 (1400)	total: 1m	remaining: 1m 9s
1600:	learn: 0.2568405	test: 0.4774344	best: 0.4772172 (1583)	total: 1m 10s	remaining: 1m 1s
1800:	learn: 0.2462092	test: 0.4762278	best: 0.4761707 (1795)	total: 1m 20s	remaining: 53.3s
2000:	learn: 0.2363777	test: 0.4751213	best: 0.4751213 (2000)	total: 1m 29s	remaining: 44.8

,fold,val_rmse,train_rmse,gap,best_iteration
0,1,0.474951,0.234124,0.240828,2055
1,2,0.361159,0.198363,0.162797,2998
2,3,0.348900,0.197103,0.151797,2970
3,4,0.372325,0.195876,0.176449,2968
4,5,0.366275,0.198686,0.167589,2988



========== OOF RMSE 비교 ==========
Tidal 기존 l2=3  : 0.386370
Tidal 신규 l2=10 : 0.387842

전체 기존 l2=3  : 0.230576
전체 신규 l2=10 : 0.230623
전체 RMSE 변화  : +0.000047
